# Module A — 위성 AGB Nowcasting
## 다목적 산림경영 AI Agent | 충북 보은 파일럿

**목적**: 현지 조사 없이 위성 원격탐사 데이터로 임분 현재 상태 추정  
**방법**: GEDI L4A 실측 AGB + 위성 피처(Sentinel-2/SAR/PALSAR/DEM) → Quantile Random Forest  
**출력**: `StandStateEstimate` (AGB · 입목축적 · 탄소량 + 90% 예측구간)

---

| 단계 | 내용 |
|------|------|
| Step 1 | 환경 설정 및 GEE 인증 |
| Step 2 | GEDI L4A 학습 데이터 추출 (GEE) |
| Step 3 | 위성 피처 추출 (GEE) |
| Step 4 | 데이터 전처리 및 탐색 |
| Step 5 | 베이스라인 모델 (Random Forest) |
| Step 6 | 최종 모델 (Quantile RF) |
| Step 7 | 발표용 Figure 3장 |
| Step 8 | predict_stand() 함수 완성 |


---
## Step 1 — 환경 설정 및 GEE 인증


In [ ]:
# 필요 패키지 설치
import subprocess, sys

packages = [
    "earthengine-api", "geemap", "geopandas",
    "rasterio", "rioxarray", "quantile-forest",
    "pydantic", "pyproj"
]
for pkg in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg],
        capture_output=True, text=True
    )
    status = "✅" if result.returncode == 0 else "❌"
    print(f"{status} {pkg}")
print("\n설치 완료!")

In [ ]:
# GEE 인증 (최초 1회만)
import ee
ee.Authenticate()

In [ ]:
# GEE 초기화 및 연결 테스트
import ee
ee.Initialize(project='constant-goods-461116-r4')

aoi_test = ee.Geometry.Point([127.7, 36.5])
img = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
       .filterBounds(aoi_test)
       .filterDate("2023-06-01", "2023-08-31")
       .first())

print("✅ GEE 연결 성공!")
print("이미지 ID:", img.get('system:index').getInfo())
print("촬영 날짜:", img.date().format('YYYY-MM-dd').getInfo())

In [ ]:
# 보은군 경계 로드
import json, geopandas as gpd

BOEUN_PATH = r"G:\연구\공모전\ai공모전\데이터\boeun_boundary_wgs84.geojson"

gdf_boeun = gpd.read_file(BOEUN_PATH)
with open(BOEUN_PATH) as f:
    gj = json.load(f)

boeun_ee = ee.Geometry(gj['features'][0]['geometry'])
area_km2 = boeun_ee.area().divide(1e6).getInfo()

print(f"✅ 보은군 경계 로드 완료")
print(f"면적: {area_km2:.1f} km²")

---
## Step 2 — GEDI L4A 학습 데이터 추출 (GEE)

**GEDI L4A**: NASA/LARSE 우주 라이다 — 지상부 바이오매스(AGB) 실측값  
**품질 필터**: l4_quality=1, degrade=0, sensitivity>0.9, se_ratio<0.5  
**결과**: 보은군 11,026개 footprint


In [ ]:
import ee
ee.Initialize(project='constant-goods-461116-r4')

# GEDI INDEX로 보은군 sub-asset 목록 확인
index = ee.FeatureCollection("LARSE/GEDI/GEDI04_A_002_INDEX")\
          .filterBounds(boeun_ee)
table_ids = index.aggregate_array("table_id").getInfo()
print(f"GEDI sub-asset 수: {len(table_ids)}개")

# 보은군 footprint 합치기
boeun_collections = [
    ee.FeatureCollection(tid).filterBounds(boeun_ee)
    for tid in table_ids
]
boeun_merged = ee.FeatureCollection(boeun_collections).flatten()
print(f"전체 footprint: {boeun_merged.size().getInfo():,}개")

In [ ]:
# 품질 필터 적용
boeun_q = (boeun_merged
    .filter(ee.Filter.eq("l4_quality_flag", 1))
    .filter(ee.Filter.eq("degrade_flag", 0))
    .filter(ee.Filter.gt("sensitivity", 0.9))
    .filter(ee.Filter.lt("agbd", 500))
    .filter(ee.Filter.gt("agbd", 0))
    .filter(ee.Filter.gt("elev_lowestmode", 0))
    .map(lambda f: f.set('se_ratio',
         ee.Number(f.get('agbd_se')).divide(
         ee.Number(f.get('agbd')).add(0.001))))
    .filter(ee.Filter.lt('se_ratio', 0.5))
)

n = boeun_q.size().getInfo()
print(f"품질 필터 후: {n:,}개 footprint")

stats = boeun_q.aggregate_stats('agbd').getInfo()
print(f"agbd 평균: {stats['mean']:.1f} Mg/ha")
print(f"agbd std:  {stats['total_sd']:.1f} Mg/ha")

---
## Step 3 — 위성 피처 추출 (GEE)

| 데이터 | 피처 | 처리 |
|--------|------|------|
| Sentinel-2 SR | B2,B4~B8,B8A,B11,B12 + NDVI,NDRE,NBR,NDMI,EVI | CS+ 마스킹, 2023년 5~10월 중위값 |
| Sentinel-1 GRD | VV_mean,VH_mean,VV_std,VV_VH_ratio | Speckle 제거 |
| PALSAR-2 | HH_db,HV_db,HH_HV_db | 스무딩 |
| AW3D30 DEM | elev,slope,northness,eastness | — |

**피처 총 25개** (B3 제거: B2·B4와 상관 0.9+, 다중공산성)


In [ ]:
import ee
ee.Initialize(project='constant-goods-461116-r4')

# ── Sentinel-2 (CS+ 마스킹) ──────────────────────────────────
cs = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")\
       .filterBounds(boeun_ee).filterDate("2023-05-01","2023-10-31")

s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(boeun_ee)
      .filterDate("2023-05-01", "2023-10-31")
      .linkCollection(cs, ['cs_cdf'])
      .map(lambda img: img.updateMask(img.select('cs_cdf').gte(0.6)))
      .select(['B2','B4','B5','B6','B7','B8','B8A','B11','B12'])
      .median().toFloat())

ndvi = s2.normalizedDifference(['B8','B4']).rename('NDVI')
ndre = s2.normalizedDifference(['B8','B5']).rename('NDRE')
nbr  = s2.normalizedDifference(['B8','B12']).rename('NBR')
ndmi = s2.normalizedDifference(['B8','B11']).rename('NDMI')
evi  = s2.expression(
    '2.5*((NIR-R)/(NIR+6*R-7.5*B+1))',
    {'NIR':s2.select('B8'),'R':s2.select('B4'),'B':s2.select('B2')}
).rename('EVI')

# ── Sentinel-1 SAR (Speckle 제거) ────────────────────────────
s1 = (ee.ImageCollection("COPERNICUS/S1_GRD")
      .filterBounds(boeun_ee).filterDate("2023-05-01","2023-10-31")
      .filter(ee.Filter.eq("instrumentMode","IW"))
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation","VV"))
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation","VH")))

speckle = lambda img: img.focal_mean(radius=1, kernelType='square', units='pixels')
vv_mean = s1.select('VV').map(speckle).mean().rename('VV_mean')
vh_mean = s1.select('VH').map(speckle).mean().rename('VH_mean')
vv_std  = s1.select('VV').reduce(ee.Reducer.stdDev()).rename('VV_std')
vv_vh   = vv_mean.subtract(vh_mean).rename('VV_VH_ratio')

# ── ALOS-2 PALSAR ─────────────────────────────────────────────
palsar = (ee.ImageCollection("JAXA/ALOS/PALSAR/YEARLY/SAR_EPOCH")
          .filterDate("2023-01-01","2024-01-01").first())
HH    = palsar.select('HH').pow(2).log10().multiply(10).subtract(83.0).rename('HH_db')
HV    = palsar.select('HV').pow(2).log10().multiply(10).subtract(83.0).rename('HV_db')
HH_HV = HH.subtract(HV).rename('HH_HV_db')

# ── AW3D30 DEM ────────────────────────────────────────────────
dem       = (ee.ImageCollection("JAXA/ALOS/AW3D30/V3_2")
             .first().select('DSM').rename('elev'))
slope     = ee.Terrain.slope(dem).rename('slope')
aspect    = ee.Terrain.aspect(dem).multiply(3.14159/180)
northness = aspect.cos().rename('northness')
eastness  = aspect.sin().rename('eastness')

# ── 전체 피처 이미지 합치기 (25개) ────────────────────────────
img_all = ee.Image.cat([
    s2, ndvi, ndre, nbr, ndmi, evi,
    vv_mean, vh_mean, vv_std, vv_vh,
    HH, HV, HH_HV,
    dem, slope, northness, eastness
]).toFloat()

print("피처 밴드 수:", len(img_all.bandNames().getInfo()), "개")

In [ ]:
# GEDI footprint 위치에서 위성 피처 추출 (25m 버퍼)
def extract_features(feature):
    values = img_all.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=feature.geometry().buffer(25),
        scale=10,
        maxPixels=1e6
    )
    return feature.setMulti(values)

# 테스트 3개 먼저 확인
test_3 = ee.FeatureCollection(boeun_q.toList(3))
result = test_3.map(extract_features).first().getInfo()['properties']
print(f"NDVI: {result.get('NDVI'):.4f}")
print(f"elev: {result.get('elev'):.0f}m")
print(f"HV_db: {result.get('HV_db'):.4f}")
print("✅ 피처 추출 정상!")

In [ ]:
# 전체 11,026개 Export (Google Drive → GEE_exports 폴더)
task = ee.batch.Export.table.toDrive(
    collection=boeun_q.map(extract_features),
    description='training-boeun-gedi-2023',
    folder='GEE_exports',
    fileNamePrefix='boeun_gedi_training_clean',
    fileFormat='CSV'
)
task.start()
print(f"✅ Export 시작! ID: {task.id}")
print("진행 확인: https://code.earthengine.google.com/tasks")
print("완료 후 구글드라이브 → GEE_exports → boeun_gedi_training_clean.csv 다운로드")

---
## Step 4 — 데이터 전처리 및 탐색

**최종 학습 데이터**: `boeun_gedi_training_clean.csv`  
- 11,026개 GEDI footprint  
- 피처 25개 (B3 제거)  
- 라벨: agbd (Mg/ha)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 한글 폰트
for font in fm.findSystemFonts():
    if 'malgun' in font.lower() or 'nanum' in font.lower():
        fm.fontManager.addfont(font)
        plt.rcParams['font.family'] = fm.FontProperties(fname=font).get_name()
        break
plt.rcParams['axes.unicode_minus'] = False

DATA_PATH = r"G:\연구\공모전\ai공모전\데이터\boeun_gedi_training_clean.csv"

df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
print(f"원본 데이터: {len(df):,}행 × {df.shape[1]}컬럼")
print(f"결측값: {df.isnull().sum().sum()}")
print(f"\nagbd 통계:")
print(df['agbd'].describe().round(1))

In [ ]:
# NDVI >= 0.3 필터 (비산림 제거) + 샘플 가중치
df = df[df['NDVI'] >= 0.3].copy()
df['sample_weight'] = 1.0 / (df['agbd_se'] + 1.0)

print(f"NDVI 필터 후: {len(df):,}행")
print(f"agbd 평균: {df['agbd'].mean():.1f} Mg/ha")
print(f"agbd std:  {df['agbd'].std():.1f} Mg/ha")
print(f"NDVI 평균: {df['NDVI'].mean():.3f}")

In [ ]:
# 피처 정의
FEATURES = [
    'B2','B4','B5','B6','B7','B8','B8A','B11','B12',   # Sentinel-2 (9)
    'NDVI','NDRE','NBR','NDMI','EVI',                   # 식생지수 (5)
    'VV_mean','VH_mean','VV_std','VV_VH_ratio',         # Sentinel-1 (4)
    'HH_db','HV_db','HH_HV_db',                        # PALSAR (3)
    'elev','slope','northness','eastness'               # DEM (4)
]  # 총 25개

print(f"피처 수: {len(FEATURES)}개")

# AGB vs 피처 상관관계 Top 10
corr = df[FEATURES + ['agbd']].corr()['agbd'].drop('agbd')
top10 = corr.abs().sort_values(ascending=False).head(10)
print("\nAGB 상관관계 Top 10:")
for feat, val in top10.items():
    raw = corr[feat]
    bar = '█' * int(abs(raw) * 60)
    print(f"  {feat:15s}: {raw:+.3f} {bar}")

In [ ]:
from sklearn.model_selection import train_test_split

trn, val = train_test_split(df, test_size=0.15, random_state=42)
print(f"학습셋: {len(trn):,}개")
print(f"검증셋: {len(val):,}개")

---
## Step 5 — 베이스라인 모델 (Random Forest)

| 모델 | R² | RMSE |
|------|-----|------|
| RF (B3 포함) | 0.413 | 65.7 Mg/ha |
| **RF (B3 제거, NDVI≥0.3)** | **0.479** | **59.3 Mg/ha** |
| XGBoost | 0.438 | 61.7 Mg/ha |


In [ ]:
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

print("RF 학습 중...")
start = time.time()

rf = RandomForestRegressor(
    n_estimators=1000,
    max_depth=None,
    max_features=0.5,
    min_samples_leaf=5,
    min_samples_split=5,
    n_jobs=-1,
    random_state=42
)
rf.fit(trn[FEATURES], trn['agbd'],
       sample_weight=trn['sample_weight'])

print(f"학습 시간: {time.time()-start:.1f}초")

pred_val  = rf.predict(val[FEATURES])
r2_val    = r2_score(val['agbd'], pred_val)
rmse_val  = np.sqrt(mean_squared_error(val['agbd'], pred_val))

print(f"\n===== RF 베이스라인 =====")
print(f"R²:   {r2_val:.3f}")
print(f"RMSE: {rmse_val:.1f} Mg/ha")

In [ ]:
# 피처 중요도 Top 10
importances = pd.Series(rf.feature_importances_, index=FEATURES)
top10 = importances.sort_values(ascending=False).head(10)

print("피처 중요도 Top 10:")
for feat, imp in top10.items():
    bar = '█' * int(imp * 300)
    print(f"  {feat:15s} {imp:.3f} {bar}")

---
## Step 6 — 최종 모델 (Quantile Random Forest)

**선택 이유**: RF와 동일 성능 + 90% 예측구간(PI) 제공  
**90%PI coverage = 0.916** → 불확실성 정량화 가능


In [ ]:
from quantile_forest import RandomForestQuantileRegressor

print("Quantile RF 학습 중...")
start = time.time()

qrf = RandomForestQuantileRegressor(
    n_estimators=1000,
    max_features=0.5,
    min_samples_leaf=5,
    min_samples_split=5,
    n_jobs=-1,
    random_state=42
)
qrf.fit(trn[FEATURES], trn['agbd'],
        sample_weight=trn['sample_weight'])

print(f"학습 시간: {time.time()-start:.1f}초")

pred_q   = qrf.predict(val[FEATURES], quantiles=[0.05, 0.50, 0.95])
q05, q50, q95 = pred_q[:,0], pred_q[:,1], pred_q[:,2]

r2_val   = r2_score(val['agbd'], q50)
rmse_val = np.sqrt(mean_squared_error(val['agbd'], q50))
coverage = ((val['agbd'].values >= q05) & (val['agbd'].values <= q95)).mean()

print(f"\n===== Quantile RF 성능 =====")
print(f"R²:              {r2_val:.3f}")
print(f"RMSE:            {rmse_val:.1f} Mg/ha")
print(f"90%PI coverage:  {coverage:.3f}  (목표: 0.85~0.95)")

In [ ]:
# 모델 저장
import joblib

MODEL_PATH = r"G:\연구\공모전\ai공모전\데이터\qrf_model.pkl"
joblib.dump(qrf, MODEL_PATH)
print(f"✅ 모델 저장: {MODEL_PATH}")

---
## Step 7 — 발표용 Figure 3장

- **Fig 1**: GEDI 예측 성능 scatter (x=실측, y=예측)
- **Fig 2**: NFI 외부 검증 (x=NFI 실측 입목축적, y=모델 예측)
- **Fig 3**: 보은군 AGB 공간 분포 (10m raster)

> **Fig 2 한계**: R²=-0.187 — GEDI saturation으로 인한 고AGB 과소추정 (AGB>200 Mg/ha 구간)  
> 이는 모델 자체 문제가 아닌 GEDI 라이다 포화 현상(Random Forest saturation)에 기인함


In [ ]:
# ── Fig 1: GEDI 성능 scatter ─────────────────────────────────
y_true = val['agbd'].values
y_pred = q50
y_q05  = q05
y_q95  = q95

r2   = r2_score(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
cov  = ((y_true >= y_q05) & (y_true <= y_q95)).mean()

fig, ax = plt.subplots(figsize=(8, 8), dpi=150)

ax.errorbar(y_true, y_pred,
            yerr=[y_pred-y_q05, y_q95-y_pred],
            fmt='none', alpha=0.08, color='steelblue',
            elinewidth=0.5, capsize=0)

sc = ax.scatter(y_true, y_pred, c=y_pred-y_true,
                cmap='RdBu_r', vmin=-150, vmax=150,
                alpha=0.5, s=15, zorder=3)
plt.colorbar(sc, ax=ax, label='잔차 (예측 - 실측, Mg/ha)')

ax.plot([0,500],[0,500], 'k--', lw=1.5, label='1:1 line')
z = np.polyfit(y_true, y_pred, 1)
xfit = np.linspace(0, 500, 100)
ax.plot(xfit, np.poly1d(z)(xfit), 'r-', lw=1.5, label='회귀선')

ax.text(0.05, 0.95,
        f"n = {len(y_true):,}\nR² = {r2:.3f}\nRMSE = {rmse:.1f} Mg/ha",
        transform=ax.transAxes, va='top', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

ax.set_xlabel('GEDI 실측 AGB (Mg/ha)', fontsize=13)
ax.set_ylabel('모델 예측 AGB (Mg/ha)', fontsize=13)
ax.set_title(
    f'Fig 1 — GEDI AGB 예측 성능\n'
    f'R²={r2:.3f}  RMSE={rmse:.1f} Mg/ha  90%PI coverage={cov:.3f}',
    fontsize=13)
ax.set_xlim(0,520); ax.set_ylim(0,520)
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(r"G:\연구\공모전\ai공모전\데이터\fig1_gedi_scatter.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Fig 1 저장 완료")

In [ ]:
# ── Fig 2: NFI 외부 검증 ─────────────────────────────────────
# 사전 준비: GEE에서 NFI 표본점 위성 피처 추출
# (아래 GEE export 코드 실행 후 nfi_boeun_satellite_features.csv 다운로드)

import ee, pandas as pd, numpy as np
from pyproj import Transformer

ee.Initialize(project='constant-goods-461116-r4')

NFI_PATH = r"G:\연구\공모전\ai공모전\데이터\mdb_NFI_7_수정.xlsx"

df_stand = pd.read_excel(NFI_PATH, sheet_name='임분조사표')
df_boeun_nfi = df_stand[df_stand['시군구'].str.contains('보은', na=False)].copy()
df_boeun_nfi = df_boeun_nfi.dropna(subset=['좌표N','좌표E'])

tf = Transformer.from_crs("EPSG:5174","EPSG:4326", always_xy=True)
lons, lats = tf.transform(df_boeun_nfi['좌표E'].values, df_boeun_nfi['좌표N'].values)
df_boeun_nfi['lon'] = lons
df_boeun_nfi['lat'] = lats
print(f"보은군 NFI 표본점: {len(df_boeun_nfi)}개")
print(f"경도: {lons.min():.4f}~{lons.max():.4f}")
print(f"위도: {lats.min():.4f}~{lats.max():.4f}")

# GEE FeatureCollection 생성
nfi_feats = []
for _, row in df_boeun_nfi.iterrows():
    nfi_feats.append(ee.Feature(
        ee.Geometry.Point([row['lon'], row['lat']]),
        {'plot_id': str(row['표본점번호']),
         'imsan':   str(row.get('임상','')),
         'younggup':str(row.get('영급',''))}
    ))
nfi_fc = ee.FeatureCollection(nfi_feats)

# 피처 추출 및 Export
nfi_result = nfi_fc.map(lambda f: f.setMulti(
    img_all.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=f.geometry().buffer(25),
        scale=10, maxPixels=1e6
    )
))
task = ee.batch.Export.table.toDrive(
    collection=nfi_result,
    description='nfi_boeun_satellite_features',
    folder='GEE_exports',
    fileNamePrefix='nfi_boeun_satellite_features',
    fileFormat='CSV'
)
task.start()
print(f"✅ NFI Export 시작! ID: {task.id}")

In [ ]:
# NFI CSV 다운로드 후 실행
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

NFI_SAT_PATH = r"G:\연구\공모전\ai공모전\데이터\nfi_boeun_satellite_features.csv"
NFI_XLSX_PATH = r"G:\연구\공모전\ai공모전\데이터\mdb_NFI_7_수정.xlsx"

FEATURES_NO_DEM = [
    'B2','B4','B5','B6','B7','B8','B8A','B11','B12',
    'NDVI','NDRE','NBR','NDMI','EVI',
    'VV_mean','VH_mean','VV_std','VV_VH_ratio',
    'HH_db','HV_db','HH_HV_db'
]
DEM_FEATURES = ['elev','slope','northness','eastness']
FEATURES_ALL = FEATURES_NO_DEM + DEM_FEATURES

# DEM 보간 (raster에서 DEM NaN → KNN 보간)
nn_dem = NearestNeighbors(n_neighbors=5, n_jobs=-1)
nn_dem.fit(df[FEATURES_NO_DEM].values)

df_nfi_sat = pd.read_csv(NFI_SAT_PATH)
dists, idxs = nn_dem.kneighbors(df_nfi_sat[FEATURES_NO_DEM].values)
for j, dem_col in enumerate(DEM_FEATURES):
    df_nfi_sat[dem_col] = [
        np.average(df[dem_col].values[idx], weights=1/(dist+1e-6))
        for idx, dist in zip(idxs, dists)
    ]

# QRF 예측
preds = qrf.predict(df_nfi_sat[FEATURES_ALL], quantiles=[0.05,0.50,0.95])
df_nfi_sat['agb_med'] = preds[:,1]
df_nfi_sat['agb_q05'] = preds[:,0]
df_nfi_sat['agb_q95'] = preds[:,2]

SPECIES_MAP = {"침엽수림(D)":("기본침엽",0.44,1.50),
               "활엽수림(H)":("기본활엽",0.60,1.43),
               "혼효림(M)":  ("기본활엽",0.60,1.43)}
def agb2vol(agb, imsan):
    _, D, BEF = SPECIES_MAP.get(imsan, ("기본활엽",0.60,1.43))
    return agb / (D * BEF)

df_nfi_sat['vol_pred'] = df_nfi_sat.apply(
    lambda r: agb2vol(r['agb_med'], r['imsan']), axis=1)
df_nfi_sat['vol_q05']  = df_nfi_sat.apply(
    lambda r: agb2vol(r['agb_q05'], r['imsan']), axis=1)
df_nfi_sat['vol_q95']  = df_nfi_sat.apply(
    lambda r: agb2vol(r['agb_q95'], r['imsan']), axis=1)

# NFI 실측 입목축적 계산
df_tree = pd.read_excel(NFI_XLSX_PATH, sheet_name='임목조사표')
df_tree['추정간재적'] = pd.to_numeric(df_tree['추정간재적'], errors='coerce').fillna(0)
vol_plot = (df_tree.groupby('표본점번호')['추정간재적'].sum() / 0.04).reset_index()
vol_plot.columns = ['plot_id','volume_nfi']
vol_plot['plot_id'] = vol_plot['plot_id'].astype(str)

df_nfi_sat['plot_id'] = df_nfi_sat['plot_id'].astype(str)
df_fig2 = df_nfi_sat.merge(vol_plot, on='plot_id', how='inner')
df_fig2 = df_fig2[df_fig2['volume_nfi'] > 0].copy()

r2   = r2_score(df_fig2['volume_nfi'], df_fig2['vol_pred'])
rmse = np.sqrt(mean_squared_error(df_fig2['volume_nfi'], df_fig2['vol_pred']))
print(f"NFI 외부 검증 R²={r2:.3f}, RMSE={rmse:.1f} m³/ha")
print("※ GEDI saturation으로 인한 고AGB 침엽수 과소추정 반영")

# 시각화
COLOR = {"침엽수림(D)":"#2166ac","활엽수림(H)":"#d73027","혼효림(M)":"#4dac26"}
fig, ax = plt.subplots(figsize=(7,7), dpi=150)
for imsan, grp in df_fig2.groupby('imsan'):
    ax.errorbar(grp['volume_nfi'], grp['vol_pred'],
                yerr=[grp['vol_pred']-grp['vol_q05'], grp['vol_q95']-grp['vol_pred']],
                fmt='o', color=COLOR.get(imsan,'gray'),
                alpha=0.8, capsize=3, markersize=6, lw=1.0, label=imsan)
vmax = max(df_fig2['volume_nfi'].max(), df_fig2['vol_pred'].max()) * 1.1
ax.plot([0,vmax],[0,vmax],'k--',lw=1.2,label='1:1 line')
m,b = np.polyfit(df_fig2['volume_nfi'], df_fig2['vol_pred'], 1)
ax.plot(np.linspace(0,vmax,100), m*np.linspace(0,vmax,100)+b, 'r-', lw=1.5, label='회귀선')
ax.text(0.05,0.95,f"n = {len(df_fig2)}\nR² = {r2:.3f}\nRMSE = {rmse:.1f} m³/ha",
        transform=ax.transAxes, va='top', fontsize=11,
        bbox=dict(boxstyle='round',facecolor='wheat',alpha=0.8))
ax.set_xlabel("NFI 실측 입목축적 (m³/ha)", fontsize=13)
ax.set_ylabel("모델 예측 입목축적 (m³/ha)", fontsize=13)
ax.set_title(f"Fig 2 — NFI 외부 검증\nR²={r2:.3f}  RMSE={rmse:.1f} m³/ha", fontsize=13)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.set_xlim(0,vmax); ax.set_ylim(0,vmax)
plt.tight_layout()
plt.savefig(r"G:\연구\공모전\ai공모전\데이터\fig2_nfi_validation.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Fig 2 저장 완료")

In [ ]:
# ── Fig 3: 보은군 AGB 공간 분포 (10m raster) ────────────────
# 사전 준비: GEE에서 boeun_satellite_features_10m.tif export 완료 필요

import rasterio
import geopandas as gpd

TIFF_PATH = r"G:\연구\공모전\ai공모전\데이터\boeun_satellite_features_10m.tif"

# raster 읽기
with rasterio.open(TIFF_PATH) as src:
    bands   = src.read()
    profile = src.profile.copy()
    extent  = [src.bounds.left, src.bounds.right,
               src.bounds.bottom, src.bounds.top]

H, W = bands.shape[1], bands.shape[2]
pixels = bands.reshape(bands.shape[0], -1).T

# 유효 픽셀 마스크 (DEM 제외 21개 기준)
no_dem_idx = [FEATURES_ALL.index(f) for f in FEATURES_NO_DEM]
valid_mask = ~np.any(np.isnan(pixels[:, no_dem_idx]), axis=1)
print(f"유효 픽셀: {valid_mask.sum():,}개 ({valid_mask.sum()/(H*W)*100:.1f}%)")

# DEM KNN 보간
X_nodem = pixels[valid_mask][:, no_dem_idx]
BATCH = 100000
n_valid = X_nodem.shape[0]
dem_interp = np.zeros((n_valid, 4))
for i in range(0, n_valid, BATCH):
    d, ix = nn_dem.kneighbors(X_nodem[i:i+BATCH])
    for j, c in enumerate(DEM_FEATURES):
        dem_interp[i:i+BATCH, j] = [
            np.average(df[c].values[idx], weights=1/(dist+1e-6))
            for idx, dist in zip(ix, d)
        ]
    print(f"  DEM 보간: {min(i+BATCH,n_valid):,}/{n_valid:,}")

X_valid = np.hstack([X_nodem, dem_interp])

# AGB 배치 예측
BATCH2 = 50000
agb_pred = np.full(n_valid, np.nan)
agb_q05  = np.full(n_valid, np.nan)
agb_q95  = np.full(n_valid, np.nan)
for i in range(0, n_valid, BATCH2):
    batch = pd.DataFrame(X_valid[i:i+BATCH2], columns=FEATURES_ALL)
    p = qrf.predict(batch, quantiles=[0.05,0.50,0.95])
    agb_pred[i:i+BATCH2] = p[:,1]
    agb_q05[i:i+BATCH2]  = p[:,0]
    agb_q95[i:i+BATCH2]  = p[:,2]
    print(f"  예측: {min(i+BATCH2,n_valid):,}/{n_valid:,} ({min(i+BATCH2,n_valid)/n_valid*100:.0f}%)")

# 2D raster 복원
def to_map(arr):
    m = np.full(H*W, np.nan)
    m[valid_mask] = arr
    return m.reshape(H, W)

agb_map     = to_map(agb_pred)
agb_q05_map = to_map(agb_q05)
agb_q95_map = to_map(agb_q95)

valid_vals = agb_map[~np.isnan(agb_map)]
print(f"\n✅ 예측 완료!")
print(f"AGB 평균: {valid_vals.mean():.1f} Mg/ha")
print(f"AGB 최대: {valid_vals.max():.1f} Mg/ha")
print(f"산림면적: {len(valid_vals)*100/1e4:.0f} ha")

# GeoTIFF 저장
OUT_TIFF = r"G:\연구\공모전\ai공모전\데이터\boeun_agb_10m.tif"
profile.update(count=3, dtype='float32', nodata=-9999)
with rasterio.open(OUT_TIFF, 'w', **profile) as dst:
    for i, m in enumerate([agb_map, agb_q05_map, agb_q95_map], 1):
        dst.write(np.where(np.isnan(m), -9999, m).astype('float32'), i)
print(f"✅ AGB raster 저장: {OUT_TIFF}")

# 시각화
boeun_gdf = gpd.read_file(BOEUN_PATH)
fig, axes = plt.subplots(1, 3, figsize=(18, 7), dpi=150)
cmap = plt.cm.YlGn

for ax, data, title in zip(axes,
    [agb_map, agb_q05_map, agb_q95_map],
    ['AGB 중앙값 (q50)', 'AGB 하한 (q05)', 'AGB 상한 (q95)']):
    im = ax.imshow(data, cmap=cmap, vmin=0, vmax=300,
                   extent=extent, origin='upper', aspect='equal')
    boeun_gdf.boundary.plot(ax=ax, color='black', linewidth=1.2)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("경도", fontsize=9); ax.set_ylabel("위도", fontsize=9)
    plt.colorbar(im, ax=ax, label='AGB (Mg/ha)', shrink=0.8)

fig.suptitle(
    f"Fig 3 — 보은군 지상부 바이오매스 공간 분포 (2023)\n"
    f"평균 {valid_vals.mean():.1f} Mg/ha  |  "
    f"최대 {valid_vals.max():.1f} Mg/ha  |  "
    f"산림면적 {len(valid_vals)*100/1e4:.0f} ha",
    fontsize=13)
plt.tight_layout()
plt.savefig(r"G:\연구\공모전\ai공모전\데이터\fig3_agb_map.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Fig 3 저장 완료")

---
## Step 8 — predict_stand() 함수 완성

팀원 import용 최종 함수. `module_a/predict_stand.py`에 저장됨.  
희도(Module C), 수범(Module E)에서 아래와 같이 호출:

```python
from module_a.predict_stand import predict_stand
result = predict_stand(geom_wkt=..., pnu=..., species_dominant=...)
```


In [ ]:
import scipy.stats as stats
from pydantic import BaseModel, Field
from typing import Optional, Dict, Literal
from datetime import datetime

# ── 바이오매스 변환 계수 (산림과학원) ────────────────────────
SPECIES_PARAMS = {
    "잣나무":    {"D": 0.41, "BEF": 1.35, "R": 0.28, "CF": 0.49},
    "낙엽송":    {"D": 0.45, "BEF": 1.34, "R": 0.29, "CF": 0.51},
    "강원소나무": {"D": 0.42, "BEF": 1.74, "R": 0.26, "CF": 0.51},
    "중부소나무": {"D": 0.47, "BEF": 1.74, "R": 0.26, "CF": 0.51},
    "리기다":    {"D": 0.50, "BEF": 1.33, "R": 0.36, "CF": 0.51},
    "신갈":      {"D": 0.66, "BEF": 1.45, "R": 0.43, "CF": 0.47},
    "굴참":      {"D": 0.72, "BEF": 1.45, "R": 0.43, "CF": 0.47},
    "상수리":    {"D": 0.72, "BEF": 1.45, "R": 0.43, "CF": 0.47},
    "편백":      {"D": 0.41, "BEF": 1.35, "R": 0.25, "CF": 0.51},
    "자작":      {"D": 0.61, "BEF": 1.40, "R": 0.31, "CF": 0.47},
    "백합":      {"D": 0.42, "BEF": 1.40, "R": 0.34, "CF": 0.47},
    "기본침엽":  {"D": 0.44, "BEF": 1.50, "R": 0.28, "CF": 0.51},
    "기본활엽":  {"D": 0.60, "BEF": 1.43, "R": 0.38, "CF": 0.47},
}

# ── Pydantic 출력 스키마 ──────────────────────────────────────
class StandStateEstimate(BaseModel):
    pnu:               str
    geom_wkt:          str
    area_ha:           float = Field(..., gt=0)
    estimated_at:      datetime
    species_dominant:  str
    species_secondary: Optional[str]  = None
    age_estimate:      Optional[int]  = Field(None, ge=0, le=200)
    age_class:         Optional[str]  = None
    agb_mg_per_ha:     float
    agb_q05:           float
    agb_q95:           float
    volume_m3_per_ha:  float
    volume_q05:        float
    volume_q95:        float
    carbon_tc_per_ha:  float
    carbon_q05:        float
    carbon_q95:        float
    grade_distribution: Dict[str, float]
    n_gedi_footprints:  int
    n_s2_scenes:        int
    saturation_warning: bool = False
    confidence_level:   Literal["high","medium","low"]
    confidence_note:    Optional[str] = None

# ── 내부 함수들 ───────────────────────────────────────────────
def _agb_to_volume(agb, sp):
    p = SPECIES_PARAMS.get(sp, SPECIES_PARAMS["기본활엽"])
    return agb / (p["D"] * p["BEF"])

def _agb_to_carbon(agb, sp):
    p = SPECIES_PARAMS.get(sp, SPECIES_PARAMS["기본활엽"])
    return agb * (1 + p["R"]) * p["CF"]

def _age_class(age):
    return None if age is None else f"{(age//10)+1}영급"

def _grade_dist(agb):
    mean_dbh = 7.5 * (agb/50)**0.4
    shape, scale = 2.5, mean_dbh/0.89
    breaks = [0,6,12,18,24,30,42,999]
    labels = ["치수","소경","중경","대경1","대경2","대경3","초대경"]
    probs = {}
    for i, lab in enumerate(labels):
        probs[lab] = round(stats.weibull_min.cdf(breaks[i+1],shape,scale=scale)
                         - stats.weibull_min.cdf(breaks[i],shape,scale=scale), 4)
    total = sum(probs.values()) or 1.0
    return {k: round(v/total,4) for k,v in probs.items()}

print("✅ 스키마 및 변환 함수 정의 완료")

In [ ]:
import rasterio
from rasterio.mask import mask as rio_mask
from shapely.geometry import mapping
from shapely.wkt import loads as wkt_loads
import geopandas as gpd

RASTER_PATH = r"G:\연구\공모전\ai공모전\데이터\boeun_satellite_features_10m.tif"

def predict_stand(
    geom_wkt:          str,
    pnu:               str,
    species_dominant:  str,
    species_secondary: Optional[str] = None,
    age_estimate:      Optional[int] = None,
    n_gedi_footprints: int = 11026,
    n_s2_scenes:       int = 23,
) -> StandStateEstimate:
    # 1. 면적 계산
    geom    = wkt_loads(geom_wkt)
    area_ha = max(gpd.GeoSeries([geom], crs='EPSG:4326')
                  .to_crs('EPSG:5179').area.values[0] / 10000, 0.0001)

    # 2. raster에서 위성 피처 추출
    try:
        with rasterio.open(RASTER_PATH) as src:
            out_image, _ = rio_mask(src, [mapping(geom)], crop=True, nodata=np.nan)
        pixels   = out_image.reshape(out_image.shape[0], -1).T
        df_pix   = pd.DataFrame(pixels, columns=FEATURES_ALL)
        valid    = (~df_pix[FEATURES_NO_DEM].isnull().any(axis=1) &
                    (df_pix['NDVI'] >= 0.3))
        df_valid = df_pix[valid].copy()
    except Exception:
        df_valid = pd.DataFrame()

    n_pixels = len(df_valid)

    # 3. AGB 예측
    if n_pixels == 0:
        confidence = "low"
        note = "폴리곤 내 유효 픽셀 없음 → 학습 데이터 평균 사용"
        feat_mean = df[FEATURES_ALL].mean().values.reshape(1,-1)
        p = qrf.predict(pd.DataFrame(feat_mean, columns=FEATURES_ALL),
                        quantiles=[0.05,0.50,0.95])
        agb_q05, agb_med, agb_q95 = float(p[0,0]), float(p[0,1]), float(p[0,2])
    else:
        d, ix = nn_dem.kneighbors(df_valid[FEATURES_NO_DEM].values)
        for j, c in enumerate(DEM_FEATURES):
            df_valid[c] = [np.average(df[c].values[i], weights=1/(dist+1e-6))
                           for i, dist in zip(ix, d)]
        p = qrf.predict(df_valid[FEATURES_ALL], quantiles=[0.05,0.50,0.95])
        agb_q05 = float(np.percentile(p[:,0], 50))
        agb_med = float(np.percentile(p[:,1], 50))
        agb_q95 = float(np.percentile(p[:,2], 50))
        pi_w = agb_q95 - agb_q05
        if n_pixels >= 50 and pi_w < 100:
            confidence, note = "high", f"유효 픽셀 {n_pixels:,}개 기반 예측"
        elif n_pixels >= 10:
            confidence, note = "medium", f"유효 픽셀 {n_pixels:,}개 (중간 신뢰도)"
        else:
            confidence, note = "low", f"유효 픽셀 {n_pixels:,}개 (소수 픽셀)"

    sp = species_dominant if species_dominant in SPECIES_PARAMS else "기본활엽"

    return StandStateEstimate(
        pnu=pnu, geom_wkt=geom_wkt,
        area_ha=round(area_ha,4), estimated_at=datetime.utcnow(),
        species_dominant=species_dominant, species_secondary=species_secondary,
        age_estimate=age_estimate, age_class=_age_class(age_estimate),
        agb_mg_per_ha=round(agb_med,2), agb_q05=round(agb_q05,2), agb_q95=round(agb_q95,2),
        volume_m3_per_ha=round(_agb_to_volume(agb_med,sp),2),
        volume_q05=round(_agb_to_volume(agb_q05,sp),2),
        volume_q95=round(_agb_to_volume(agb_q95,sp),2),
        carbon_tc_per_ha=round(_agb_to_carbon(agb_med,sp),2),
        carbon_q05=round(_agb_to_carbon(agb_q05,sp),2),
        carbon_q95=round(_agb_to_carbon(agb_q95,sp),2),
        grade_distribution=_grade_dist(agb_med),
        n_gedi_footprints=n_gedi_footprints, n_s2_scenes=n_s2_scenes,
        saturation_warning=(agb_med>130 and
            species_dominant in ["잣나무","낙엽송","강원소나무","중부소나무","리기다","기본침엽"]),
        confidence_level=confidence, confidence_note=note,
    )

print("✅ predict_stand() 함수 정의 완료")

In [ ]:
# ── 테스트 실행 ──────────────────────────────────────────────
TEST_WKT = "POLYGON((127.72 36.49, 127.725 36.49, 127.725 36.495, 127.72 36.495, 127.72 36.49))"

result = predict_stand(
    geom_wkt=TEST_WKT,
    pnu="4374010100100010000",
    species_dominant="신갈",
    species_secondary="굴참",
    age_estimate=45,
    n_gedi_footprints=12,
    n_s2_scenes=8,
)

print("=" * 50)
print("StandStateEstimate 최종 출력")
print("=" * 50)
for k, v in result.model_dump().items():
    print(f"  {k:22s}: {v}")

---
## 완료 ✅

| 항목 | 결과 |
|------|------|
| 학습 데이터 | GEDI L4A 11,026개 (보은군) |
| 피처 | 위성 25개 (Sentinel-2/SAR/PALSAR/DEM) |
| 모델 | Quantile RF — R²=0.471, RMSE=59.8 Mg/ha |
| 90%PI coverage | 0.916 |
| NFI 외부검증 | R²=-0.187 (GEDI saturation 한계) |
| 보은군 산림면적 | 69,208 ha |
| 보은군 AGB 평균 | 103.8 Mg/ha |

**Module A → Module C (Faustmann NPV) 연동:**
```python
from module_a.predict_stand import predict_stand
result = predict_stand(geom_wkt=..., pnu=..., species_dominant=...)
# result.volume_m3_per_ha, result.carbon_tc_per_ha → Module C 입력
```
